# Extract VectorNet vector detections (raw DwC-A)

Downloads the **latest** VectorNet observations Darwin Core Archive from the GBIF Sweden IPT and **overwrites** the local raw copy under `data/01_raw/vector/`.

- Source: [VectorNet observations](https://www.gbif.org/dataset/4abd984b-122c-44a0-8c92-b37e2f5299b1) (`vectornet_historical_data`)
- GBIF UUID: `4abd984b-122c-44a0-8c92-b37e2f5299b1`
- License: CC-BY 4.0
- No GBIF account required

This notebook stops at raw validation (file list + occurrence smoke check). It does not clean or process the data.

In [1]:
from __future__ import annotations

import hashlib
import json
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

from tqdm import tqdm

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / "data" / "raw" / "vector"
ZIP_NAME = "vectornet_dwca.zip"
ZIP_PATH = RAW_DIR / ZIP_NAME
MANIFEST_PATH = RAW_DIR / "manifest.json"

SOURCE_URL = "https://www.gbif.se/ipt/archive.do?r=vectornet_historical_data"
RESOURCE = "vectornet_historical_data"
GBIF_DATASET_UUID = "4abd984b-122c-44a0-8c92-b37e2f5299b1"
CITATION = (
    "VectorNet, European Centre for Disease Prevention and Control, "
    "European Food Safety Authority (2025). VectorNet observations. "
    "Version 1.5. VectorNet. Occurrence dataset (CC-BY 4.0). "
    "https://doi.org/10.15468/f3k8r9"
)
DOWNLOAD_TIMEOUT_S = 300
CHUNK_SIZE = 1024 * 256

RAW_DIR, SOURCE_URL

(PosixPath('/workspaces/vector-guard/notebooks/01_sources/data/raw/vector'),
 'https://www.gbif.se/ipt/archive.do?r=vectornet_historical_data')

## Wipe local raw directory

Every run replaces the previous local copy with the archive fetched now.

In [2]:
def wipe_raw_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    for child in path.iterdir():
        if child.name == ".gitkeep":
            continue
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    (path / ".gitkeep").touch(exist_ok=True)


wipe_raw_dir(RAW_DIR)
print(f"Wiped and recreated: {RAW_DIR}")

Wiped and recreated: /workspaces/vector-guard/notebooks/01_sources/data/raw/vector


## Download latest DwC-A from IPT

In [3]:
def download_file(url: str, dest: Path, timeout_s: int = DOWNLOAD_TIMEOUT_S) -> int:
    request = Request(url, headers={"User-Agent": "vector-guard/0.1"})
    with urlopen(request, timeout=timeout_s) as response:
        total = response.headers.get("Content-Length")
        total_bytes = int(total) if total and total.isdigit() else None
        written = 0
        with dest.open("wb") as out, tqdm(
            total=total_bytes,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=dest.name,
        ) as bar:
            while True:
                chunk = response.read(CHUNK_SIZE)
                if not chunk:
                    break
                out.write(chunk)
                written += len(chunk)
                bar.update(len(chunk))
    return written


zip_bytes = download_file(SOURCE_URL, ZIP_PATH)
print(f"Downloaded {zip_bytes:,} bytes -> {ZIP_PATH}")

vectornet_dwca.zip: 6.38MB [00:06, 1.06MB/s]

Downloaded 6,684,735 bytes -> /workspaces/vector-guard/notebooks/01_sources/data/raw/vector/vectornet_dwca.zip


## Extract archive and write manifest

In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()


with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(RAW_DIR)

extracted = sorted(
    p.relative_to(RAW_DIR).as_posix()
    for p in RAW_DIR.rglob("*")
    if p.is_file() and p.name != ZIP_NAME
)

manifest = {
    "source_url": SOURCE_URL,
    "resource": RESOURCE,
    "gbif_dataset_uuid": GBIF_DATASET_UUID,
    "fetched_at": datetime.now(timezone.utc).isoformat(),
    "zip_name": ZIP_NAME,
    "zip_bytes": ZIP_PATH.stat().st_size,
    "sha256": sha256_file(ZIP_PATH),
    "citation": CITATION,
    "extracted_files": extracted,
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Extracted {len(extracted)} files")
print(f"Wrote {MANIFEST_PATH}")
manifest

Extracted 4 files
Wrote /workspaces/vector-guard/notebooks/01_sources/data/raw/vector/manifest.json


{'source_url': 'https://www.gbif.se/ipt/archive.do?r=vectornet_historical_data',
 'resource': 'vectornet_historical_data',
 'gbif_dataset_uuid': '4abd984b-122c-44a0-8c92-b37e2f5299b1',
 'fetched_at': '2026-11-13T09:33:07.849597+00:00',
 'zip_name': 'vectornet_dwca.zip',
 'zip_bytes': 6684735,
 'sha256': 'd42375422c9e63f392f55b80c1572632a7171fb5ea1b44326d47e2e03453ffa8',
 'citation': 'VectorNet, European Centre for Disease Prevention and Control, European Food Safety Authority (2025). VectorNet observations. Version 1.5. VectorNet. Occurrence dataset (CC-BY 4.0). https://doi.org/10.15468/f3k8r9',
 'extracted_files': ['.gitkeep', 'eml.xml', 'meta.xml', 'occurrence.txt']}

## Smoke check

Load the core occurrence table via the shared `load_vector_detections` helper.

In [5]:
from vector_guard.data import load_vector_detections, resolve_vector_core_path

print("Files in raw dir:")
for rel in sorted(p.relative_to(RAW_DIR).as_posix() for p in RAW_DIR.rglob("*") if p.is_file()):
    print(f"  - {rel}")

core_path = resolve_vector_core_path(RAW_DIR)
df = load_vector_detections(RAW_DIR)
print(f"\nCore file: {core_path.name}")
print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}): {list(df.columns)[:12]}{'...' if len(df.columns) > 12 else ''}")
df.head(10).T

Files in raw dir:
  - .gitkeep
  - eml.xml
  - manifest.json
  - meta.xml
  - occurrence.txt
  - vectornet_dwca.zip

Core file: occurrence.txt
Rows: 120,158
Columns (18): ['id', 'bibliographicCitation', 'basisOfRecord', 'occurrenceID', 'individualCount', 'lifeStage', 'associatedTaxa', 'eventDate', 'samplingProtocol', 'sampleSizeValue', 'sampleSizeUnit', 'higherGeographyID']...


,0,1,2,3,4,5,6,7,8,9
id,1141be2c-6c03-4685-922b-d7e73d77bd0d,3768ff90-473f-450d-9af1-3ec808691f17,fcb07c9a-a904-4dbf-9549-ebd9a5ba92aa,88d2c677-9895-4d5d-9311-ac6491c175e1,42e04dd7-fac1-43dd-a8dc-2460f669302a,94ec7b26-cf79-4496-894d-101a216d9952,a05cc859-1b03-4d95-8975-ced548d20501,73b7b9a4-08d2-40f3-9d73-4e4d034981e1,115abf54-d72f-4d32-bded-a948f4a72b26,f578f41b-6fbf-4210-b8e3-15a73e712632
bibliographicCitation,"Wilhelmsson et al,2021. Three Babesia species ...","Wilhelmsson et al,2021. Three Babesia species ...","Wilhelmsson et al,2021. Three Babesia species ...","Borsan et al,2021. High Diversity, Prevalence,...","Vikentjeva et al.,2021. Rickettsia spp. in rod...","Borsan et al,2021. High Diversity, Prevalence,...","Vikentjeva et al.,2021. Rickettsia spp. in rod...","Vikentjeva et al.,2021. Rickettsia spp. in rod...","Vikentjeva et al.,2021. Rickettsia spp. in rod...","Borsan et al,2021. High Diversity, Prevalence,..."
basisOfRecord,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation,humanObservation
occurrenceID,1141be2c-6c03-4685-922b-d7e73d77bd0d,3768ff90-473f-450d-9af1-3ec808691f17,fcb07c9a-a904-4dbf-9549-ebd9a5ba92aa,88d2c677-9895-4d5d-9311-ac6491c175e1,42e04dd7-fac1-43dd-a8dc-2460f669302a,94ec7b26-cf79-4496-894d-101a216d9952,a05cc859-1b03-4d95-8975-ced548d20501,73b7b9a4-08d2-40f3-9d73-4e4d034981e1,115abf54-d72f-4d32-bded-a948f4a72b26,f578f41b-6fbf-4210-b8e3-15a73e712632
individualCount,0,0,4,4,75,5,83,140,32,5
lifeStage,Larvae,Adults,Nymph/Pupa,Larvae,NaN,Larvae,NaN,NaN,NaN,Larvae
associatedTaxa,Acrocephalus palustris,Acrocephalus palustris,Acrocephalus palustris,Apodemus agrarius,Apodemus flavicollis,Apodemus flavicollis,Apodemus flavicollis,Apodemus flavicollis,Apodemus flavicollis,Apodemus flavicollis
eventDate,2009-03-15/2009-11-15,2009-03-15/2009-11-15,2009-03-15/2009-11-15,2018-03-01/2018-11-30,2012-04-01/2014-11-30,2018-03-01/2018-11-30,2012-04-01/2014-11-30,2012-04-01/2014-11-30,2013-04-01/2014-11-30,2018-03-01/2018-11-30
samplingProtocol,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts,Collection from live/dead hosts
sampleSizeValue,21,21,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
